# Primer Design Workshop

Welcome! In this notebook you will design PCR primers for a region of the *Drosophila melanogaster* genome.

### What is a primer?
A primer is a short piece of single-stranded DNA (usually 18–25 bases long) that acts as a starting point for DNA copying. In PCR (Polymerase Chain Reaction), you need **two primers** — one on each strand — to define the region you want to amplify.

### What will we do?
1. **Generate primer candidates**: extract a region from the *Drosophila* genome and produce every possible short sequence ("kmer") that could serve as a primer.
2. **Filter by quality**: keep only the primers with a good GC content and melting temperature (Tm).
3. **Explore the results**: inspect the surviving candidates and pick a pair.

---

## Step 0 — Install dependencies and load the module

Run this cell once at the start of your session. It installs the Python libraries we need and downloads our primer design module from GitHub.

In [ ]:
# Install required Python libraries
# biopython  → melting temperature calculation
# requests   → contact the Ensembl database over the internet
!pip install --quiet biopython requests

# Download the primers.py module from GitHub
# TODO: replace the URL below with your actual GitHub raw URL once you push the repo
# Example: https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main/primers.py
GITHUB_RAW_URL = "https://raw.githubusercontent.com/YOUR_USERNAME/YOUR_REPO/main/primers.py"

import urllib.request
urllib.request.urlretrieve(GITHUB_RAW_URL, "primers.py")
print("Module downloaded successfully!")

In [ ]:
# Import the two main functions from our module
from primers import design_primers, analyze_primers

print("Ready to design primers!")

---
## Step 1 — Choose a genomic region and generate primer candidates

We will look at a region in the *Drosophila* genome on chromosome **2L**.

The `design_primers()` function will:
- Contact Ensembl to download just the region we need (no full genome download!)
- Slide a window across the sequence to extract every possible 20, 21, 22 and 23 bp primer
- Also generate the reverse complement of each primer (needed for the second PCR primer)

In [ ]:
# ---------------------------------------------------------------
# Choose your region here!
# For Drosophila, chromosome names are: 2L, 2R, 3L, 3R, X, 4
# ---------------------------------------------------------------
CHROMOSOME = "2L"
START      = 5000    # start position (1-based)
END        = 6000    # end position   (1-based)

# Generate primer candidates
# kmer_lengths: which primer sizes to try (in base pairs)
candidates = design_primers(
    chrom       = CHROMOSOME,
    start       = START,
    end         = END,
    kmer_lengths = [20, 21, 22, 23],
    include_reverse_complement = True,
)

print(f"\nFirst primer candidate:")
print(candidates[0])

In [ ]:
# Let's look at a few candidates to understand the structure
print(f"Total primer candidates: {len(candidates)}")
print()
print("First 3 forward-strand candidates:")
for p in candidates[:6:2]:   # every other one (skip RC)
    print(f"  {p.sequence}   length={p.length}bp   {p.chrom}:{p.start}-{p.end}   strand={p.strand}")

print()
print("Their reverse complements:")
for p in candidates[1:7:2]:  # the RC versions
    print(f"  {p.sequence}   length={p.length}bp   {p.chrom}:{p.start}-{p.end}   strand={p.strand}")

---
## Step 2 — Filter primers by GC content and melting temperature (Tm)

Not every primer candidate is a good primer. We filter by two criteria:

| Property | What it measures | Good range |
|---|---|---|
| **GC content** | Fraction of G and C bases (%) | 40 – 60% |
| **Melting temp (Tm)** | Temperature where primer detaches from DNA (°C) | ~51 – 57°C |

Primers with GC content too low melt too easily. Too high and they may stick to the wrong places.
Tm should be similar between your forward and reverse primer so PCR works efficiently.

In [ ]:
# Filter candidates by GC content and Tm
good_primers = analyze_primers(
    candidates,

    # GC content thresholds (%)
    gc_min = 40.0,
    gc_max = 60.0,

    # Melting temperature thresholds (°C)
    tm_min = 51.0,
    tm_max = 56.5,

    # --- PCR reaction conditions (mM unless noted) ---
    # These affect the Tm calculation.
    # The defaults below match typical PCR conditions.
    Na    = 50.0,   # sodium
    Mg    = 2.0,    # magnesium
    K     = 0.0,    # potassium
    Tris  = 0.0,    # Tris buffer
    dNTPs = 0.2,    # nucleotides
    dnac1 = 0.016,  # primer concentration (nM)
)

---
## Step 3 — Explore the results

Let's look at the primers that passed, and see how to pick a forward/reverse pair.

In [ ]:
# Print the first 10 passing primers in a readable way
print(f"{'Sequence':<25} {'Len':>4} {'GC%':>5} {'Tm(°C)':>7}  {'Coord':<30} {'Strand'}")
print("-" * 85)
for p in good_primers[:10]:
    coord = f"{p.chrom}:{p.start}-{p.end}"
    print(f"{p.sequence:<25} {p.length:>4} {p.gc_content:>5} {p.tm:>7}  {coord:<30} {p.strand}")

In [ ]:
# Separate forward and reverse primers
forward_primers = [p for p in good_primers if p.strand == "+"]
reverse_primers = [p for p in good_primers if p.strand == "-"]

print(f"Forward primers (+): {len(forward_primers)}")
print(f"Reverse primers (-): {len(reverse_primers)}")

print("\nFirst few forward primers:")
for p in forward_primers[:5]:
    print(f"  5'-{p.sequence}-3'   Tm={p.tm}°C   GC={p.gc_content}%   pos={p.chrom}:{p.start}")

print("\nFirst few reverse primers:")
for p in reverse_primers[:5]:
    print(f"  5'-{p.sequence}-3'   Tm={p.tm}°C   GC={p.gc_content}%   pos={p.chrom}:{p.end}")

In [ ]:
# How to pick a primer pair:
# A good PCR pair needs:
#   1. One forward (+) and one reverse (-) primer
#   2. The forward primer start should be BEFORE the reverse primer end
#   3. Similar Tm values (within ~2°C of each other)
#   4. The two primers should not overlap

# Let's find all valid pairs from the first 5 forward and 5 reverse primers
print(f"{'Forward sequence':<25}  {'Reverse sequence':<25}  {'Product (bp)':>12}  {'ΔTm (°C)':>9}")
print("-" * 80)

for fwd in forward_primers[:5]:
    for rev in reverse_primers[:5]:
        # Forward primer must start before reverse primer ends
        if fwd.start >= rev.end:
            continue
        # Primers must not overlap
        if fwd.end >= rev.start:
            continue
        product_size = rev.end - fwd.start + 1
        delta_tm     = abs(fwd.tm - rev.tm)
        print(f"{fwd.sequence:<25}  {rev.sequence:<25}  {product_size:>12}  {delta_tm:>9.1f}")

---
## Exercises

Try the following to deepen your understanding:

1. **Change the region**: modify `CHROMOSOME`, `START`, and `END` in Step 1 and re-run everything. How does the number of passing primers change?

2. **Change the primer lengths**: try `kmer_lengths=[18, 19, 20]` or `kmer_lengths=[24, 25]`. What happens to the Tm values?

3. **Widen the Tm filter**: change `tm_min=49, tm_max=60` in Step 2. How many more primers pass?

4. **What if you remove `include_reverse_complement=True`?** How does the output change? Can you still find a primer pair?

5. **Challenge**: write a function that, given a list of `good_primers`, automatically returns the pair with the smallest ΔTm.